Binding a fitted ``model`` makes :class:`SeqMut` model-aware: each scored mutation carries ``delta_pred`` — the change of the model prediction score (percentage points) it induces. :meth:`SeqMut.combine` applies a *named set* of point mutations to the **same** sequence and scores each combined variant once, so 2-3 mutations are evaluated together (unlike :meth:`SeqMut.mutate`, which scores each independently).

In [1]:
import itertools
import pandas as pd
import matplotlib.pyplot as plt
import aaanalysis as aa
aa.options["verbose"] = False

# Data, CPP features, and a fitted TreeModel that scores each sequence
df_seq = aa.load_dataset(name="DOM_GSEC", n=10)
labels = df_seq["label"].to_list()
sf = aa.SequenceFeature()
df_parts = sf.get_df_parts(df_seq=df_seq)
split_kws = sf.get_split_kws()
df_scales = aa.load_scales()
cpp = aa.CPP(df_parts=df_parts, split_kws=split_kws, df_scales=df_scales, verbose=False)
df_feat = cpp.run(labels=labels, n_filter=25)
X = sf.feature_matrix(features=list(df_feat["feature"]), df_parts=df_parts, df_scales=df_scales)
tm = aa.TreeModel().fit(X, labels=labels)
entry = df_seq["entry"].iloc[0]
ts = int(df_seq.set_index("entry").loc[entry, "tmd_start"])

seqm = aa.SeqMut(model=tm)
variants = pd.DataFrame({
    "entry": [entry] * 5,
    "variant": ["double", "double", "triple", "triple", "triple"],
    "pos": [ts, ts + 1, ts, ts + 2, ts + 4],
    "to_aa": ["A", "P", "A", "K", "L"],
})
df_variant = seqm.combine(df_seq=df_seq, variants=variants, df_feat=df_feat)
aa.display_df(df_variant, n_rows=10, show_shape=True)

DataFrame shape: (2, 7)


,entry,variant,n_mut,sequence_mut,delta_cpp,shift_score,delta_pred
1,Q14802,L37A+V39K+G41L,3,MQKVTLGLLVFLAGF...PGETPPLITPGSAQS,0.326320,-0.055680,4.500000
2,Q14802,L37A+Q38P,2,MQKVTLGLLVFLAGF...PGETPPLITPGSAQS,0.066000,0.066000,2.000000


The junction-membrane-domain lengths ``jmd_n_len`` / ``jmd_c_len`` set how many flanking residues are used when rebuilding the sequence parts.

In [2]:
df_variant2 = seqm.combine(df_seq=df_seq, variants=variants, df_feat=df_feat,
                            jmd_n_len=10, jmd_c_len=10)
aa.display_df(df_variant2, n_rows=10, show_shape=True)

DataFrame shape: (2, 7)


,entry,variant,n_mut,sequence_mut,delta_cpp,shift_score,delta_pred
1,Q14802,L37A+V39K+G41L,3,MQKVTLGLLVFLAGF...PGETPPLITPGSAQS,0.326320,-0.055680,4.500000
2,Q14802,L37A+Q38P,2,MQKVTLGLLVFLAGF...PGETPPLITPGSAQS,0.066000,0.066000,2.000000
